# 按品种与交易日读取主力合约分钟行情

只读正式 silver。按 `b01_main_continuous_daily` 的日频规则，用前一交易日成交量识别当日主力，
再读取该合约的原始 1 分钟行情。区间按 `trading_date`（交易日）筛选；夜盘的 `bar_at` 可以落在前一个自然日。
不拼接复权价格，不调用行情 API，也不写入 gold/silver。

使用方式：修改下一单元格的交易所、品种、起止交易日，依次运行全部单元格。
`main_contract_daily_df` 是逐交易日主力及判定证据；`main_minute_df` 是返回的分钟行情。

In [1]:
EXCHANGE_CODE = "XSGE"
UNDERLYING_CODE = "RB"
START_DATE = "2015-01-01"
END_DATE = "2026-09-01"
ROLL_TRIGGER_RATIO = 1.10

## 1. 项目环境与权威数据契约

In [2]:
from __future__ import annotations

import math
import pathlib
import sys
from datetime import date

import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
from IPython.display import display

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录；请从 E:\\Latitude_Analytics_v2 内启动 Notebook。")

from config.data_contracts import (  # noqa: E402
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    FUTURES_DAILY_SCHEMA,
    FUTURES_MINUTE_SCHEMA,
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    arrow_to_pandas,
    validate_arrow_table,
)
from config.settings import settings  # noqa: E402

SILVER_ROOT = settings.futures_lake_root / "silver"
SILVER_SCHEMAS = (
    FUTURES_VARIETY_CALENDAR_SCHEMA,
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    FUTURES_DAILY_SCHEMA,
    FUTURES_MINUTE_SCHEMA,
)
SILVER_TABLE_INFO = {
    schema.metadata[b"table_name"].decode("utf-8"): {
        "schema": schema,
        "primary_key": schema.metadata[b"primary_key"].decode("utf-8").split(","),
        "partition_columns": schema.metadata[b"partition_columns"].decode("utf-8").split(","),
    }
    for schema in SILVER_SCHEMAS
}


def read_silver(table_name: str, filter_expression: ds.Expression) -> pd.DataFrame:
    table_info = SILVER_TABLE_INFO[table_name]
    schema = table_info["schema"]
    table_path = SILVER_ROOT / table_name
    if not table_path.is_dir():
        raise FileNotFoundError(f"silver 数据集不存在：{table_path}")
    partition_schema = pa.schema(
        [schema.field(name) for name in table_info["partition_columns"]]
    )
    dataset = ds.dataset(
        table_path,
        format="parquet",
        schema=schema,
        partitioning=ds.partitioning(partition_schema, flavor="hive"),
    )
    table = dataset.to_table(columns=schema.names, filter=filter_expression)
    return arrow_to_pandas(validate_arrow_table(table, schema), schema)


def parse_request_dates(start_date: str, end_date: str) -> tuple[date, date]:
    try:
        start = date.fromisoformat(start_date)
        end = date.fromisoformat(end_date)
    except ValueError as error:
        raise ValueError("START_DATE 和 END_DATE 必须是 YYYY-MM-DD。") from error
    if start > end:
        raise ValueError("START_DATE 不得晚于 END_DATE。")
    return start, end

## 2. 使用完整历史识别日级主力

与 `b01` 一致：信号日成交量只决定下一交易日；初始并列按持仓量、较近退市日、合约代码排序；
后续只向退市日更晚的合约滚动，默认要求挑战者成交量严格超过现主力的 1.10 倍。
若现主力退市且没有有效后月候选，该交易日不生成主力映射。

In [3]:
def load_daily_inputs(
    exchange_code: str, underlying_code: str, end_date: date
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    variety_filter = (
        (ds.field("exchange_code") == exchange_code)
        & (ds.field("year") <= end_date.year)
        & (ds.field("underlying_code") == underlying_code)
        & (ds.field("trading_date") <= end_date)
    )
    fact_filter = variety_filter
    variety_calendar_df = read_silver("dim_futures_variety_calendar", variety_filter)
    contract_calendar_df = read_silver("dim_futures_contract_calendar", variety_filter)
    daily_df = read_silver("fact_futures_daily", fact_filter)
    return variety_calendar_df, contract_calendar_df, daily_df


def build_main_contract_daily(
    variety_calendar_df: pd.DataFrame,
    contract_calendar_df: pd.DataFrame,
    daily_df: pd.DataFrame,
    roll_trigger_ratio: float,
) -> pd.DataFrame:
    if not math.isfinite(roll_trigger_ratio) or roll_trigger_ratio <= 1:
        raise ValueError("ROLL_TRIGGER_RATIO 必须是大于 1 的有限数。")

    output_columns = [
        "exchange_code", "underlying_code", "trading_date", "signal_trading_date",
        "main_contract_code", "previous_main_contract_code", "continuity_segment",
        "is_roll", "selection_reason", "main_contract_signal_volume",
        "incumbent_signal_volume", "challenger_contract_code",
        "challenger_signal_volume", "roll_trigger_ratio",
    ]
    if variety_calendar_df.empty:
        return pd.DataFrame(columns=output_columns)

    key_columns = ["exchange_code", "underlying_code", "trading_date"]
    trading_dates = sorted(variety_calendar_df["trading_date"].drop_duplicates().tolist())
    contract_dates_df = (
        contract_calendar_df[
            ["contract_code", *key_columns, "delist_date", "session_number"]
        ]
        .sort_values([*key_columns, "contract_code", "session_number"])
        .drop_duplicates([*key_columns, "contract_code"], keep="first")
        .drop(columns="session_number")
    )
    daily_work_df = daily_df[
        ["contract_code", *key_columns, "volume", "open_interest", "has_market_data"]
    ].drop_duplicates([*key_columns, "contract_code"], keep="last")
    candidate_history_df = daily_work_df.merge(
        contract_dates_df,
        how="left",
        on=["contract_code", *key_columns],
        validate="one_to_one",
    )
    ranked_df = candidate_history_df.loc[
        candidate_history_df["has_market_data"].fillna(False)
        & candidate_history_df["volume"].notna()
        & (candidate_history_df["volume"] > 0)
    ].copy()
    ranked_df["open_interest_rank"] = pd.to_numeric(
        ranked_df["open_interest"], errors="coerce"
    ).fillna(-math.inf)
    ranked_df = ranked_df.sort_values(
        ["trading_date", "volume", "open_interest_rank", "delist_date", "contract_code"],
        ascending=[True, False, False, True, True],
        kind="mergesort",
    )
    ranked_candidates_by_date = {
        trading_date: group_df.to_dict("records")
        for trading_date, group_df in ranked_df.groupby("trading_date", sort=False)
    }
    delist_lookup = contract_dates_df.groupby("contract_code")["delist_date"].max().to_dict()
    active_codes_by_date = {
        trading_date: set(group_df["contract_code"].astype(str))
        for trading_date, group_df in contract_dates_df.groupby("trading_date")
    }

    rows = []
    main_contract_code = None
    continuity_segment = 0
    for signal_index, signal_trading_date in enumerate(trading_dates[:-1]):
        trading_date = trading_dates[signal_index + 1]
        ranked_candidates = ranked_candidates_by_date.get(signal_trading_date, [])
        target_active_codes = active_codes_by_date.get(trading_date, set())
        eligible_candidates = [
            candidate for candidate in ranked_candidates
            if str(candidate["contract_code"]) in target_active_codes
        ]
        previous_main_contract_code = main_contract_code
        incumbent_signal_volume = None
        challenger_contract_code = None
        challenger_signal_volume = None

        if main_contract_code is None:
            if not eligible_candidates:
                continue
            continuity_segment += 1
            main_contract_code = str(eligible_candidates[0]["contract_code"])
            selection_reason = "bootstrap_previous_day_volume_leader"
            later_candidates = [
                candidate for candidate in eligible_candidates
                if str(candidate["contract_code"]) != main_contract_code
            ]
            if later_candidates:
                challenger_contract_code = str(later_candidates[0]["contract_code"])
                challenger_signal_volume = float(later_candidates[0]["volume"])
        else:
            incumbent_candidate = next(
                (candidate for candidate in ranked_candidates
                 if str(candidate["contract_code"]) == main_contract_code),
                None,
            )
            if incumbent_candidate is not None:
                incumbent_signal_volume = float(incumbent_candidate["volume"])
            incumbent_delist_date = delist_lookup.get(main_contract_code)
            challenger_candidates = [
                candidate for candidate in eligible_candidates
                if str(candidate["contract_code"]) != main_contract_code
                and (
                    incumbent_delist_date is None
                    or (
                        pd.notna(candidate["delist_date"])
                        and candidate["delist_date"] > incumbent_delist_date
                    )
                )
            ]
            if challenger_candidates:
                challenger_contract_code = str(challenger_candidates[0]["contract_code"])
                challenger_signal_volume = float(challenger_candidates[0]["volume"])

            if main_contract_code not in target_active_codes:
                if challenger_contract_code is None:
                    main_contract_code = None
                    continue
                main_contract_code = challenger_contract_code
                selection_reason = "incumbent_not_active_next_trading_date"
            elif incumbent_signal_volume is None:
                if challenger_contract_code is None:
                    selection_reason = "no_valid_volume_keep_incumbent"
                else:
                    main_contract_code = challenger_contract_code
                    selection_reason = "incumbent_market_data_missing"
            elif (
                challenger_signal_volume is not None
                and challenger_signal_volume > incumbent_signal_volume * roll_trigger_ratio
            ):
                main_contract_code = challenger_contract_code
                selection_reason = "challenger_volume_threshold"
            else:
                selection_reason = "incumbent_retained"

        selected_candidate = next(
            (candidate for candidate in ranked_candidates
             if str(candidate["contract_code"]) == main_contract_code),
            None,
        )
        rows.append({
            "exchange_code": str(variety_calendar_df["exchange_code"].iloc[0]),
            "underlying_code": str(variety_calendar_df["underlying_code"].iloc[0]),
            "trading_date": trading_date,
            "signal_trading_date": signal_trading_date,
            "main_contract_code": main_contract_code,
            "previous_main_contract_code": previous_main_contract_code,
            "continuity_segment": continuity_segment,
            "is_roll": previous_main_contract_code is not None
            and main_contract_code != previous_main_contract_code,
            "selection_reason": selection_reason,
            "main_contract_signal_volume": None if selected_candidate is None
            else float(selected_candidate["volume"]),
            "incumbent_signal_volume": incumbent_signal_volume,
            "challenger_contract_code": challenger_contract_code,
            "challenger_signal_volume": challenger_signal_volume,
            "roll_trigger_ratio": float(roll_trigger_ratio),
        })

    mapping_df = pd.DataFrame(rows, columns=output_columns)
    if mapping_df.duplicated(key_columns).any():
        raise ValueError("主力映射存在重复交易日。")
    return mapping_df

## 3. 分区读取当日主力的分钟行情

每次只读请求范围内的品种年月分区，并以 `trading_date + contract_code` 精确匹配日级主力。
若某个主力交易日无分钟记录，保留在日级映射表中，不制造缺失 bar。

In [4]:
def month_pairs(start_date: date, end_date: date) -> list[tuple[int, int]]:
    pairs = []
    year, month = start_date.year, start_date.month
    while (year, month) <= (end_date.year, end_date.month):
        pairs.append((year, month))
        year, month = (year + 1, 1) if month == 12 else (year, month + 1)
    return pairs


def load_main_minute_bars(
    exchange_code: str,
    underlying_code: str,
    start_date: str,
    end_date: str,
    roll_trigger_ratio: float = 1.10,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    requested_start, requested_end = parse_request_dates(start_date, end_date)
    exchange_code = exchange_code.strip().upper()
    underlying_code = underlying_code.strip().upper()
    if not exchange_code or not underlying_code:
        raise ValueError("必须指定交易所和品种代码。")

    variety_calendar_df, contract_calendar_df, daily_df = load_daily_inputs(
        exchange_code, underlying_code, requested_end
    )
    if variety_calendar_df.empty:
        raise ValueError(f"品种日历没有 {exchange_code}.{underlying_code} 的交易日。")
    all_mapping_df = build_main_contract_daily(
        variety_calendar_df, contract_calendar_df, daily_df, roll_trigger_ratio
    )
    mapping_df = all_mapping_df.loc[
        (all_mapping_df["trading_date"] >= requested_start)
        & (all_mapping_df["trading_date"] <= requested_end)
    ].copy()
    if mapping_df.empty:
        raise ValueError("所选范围没有可识别的主力合约；请检查日期、日线覆盖和流动性。")

    minute_frames = []
    for year, month in month_pairs(requested_start, requested_end):
        month_mapping_df = mapping_df.loc[
            (mapping_df["trading_date"].map(lambda day: day.year) == year)
            & (mapping_df["trading_date"].map(lambda day: day.month) == month)
        ]
        if month_mapping_df.empty:
            continue
        selected_codes = month_mapping_df["main_contract_code"].unique().tolist()
        minute_filter = (
            (ds.field("exchange_code") == exchange_code)
            & (ds.field("underlying_code") == underlying_code)
            & (ds.field("year") == year)
            & (ds.field("month") == month)
            & (ds.field("trading_date") >= requested_start)
            & (ds.field("trading_date") <= requested_end)
            & ds.field("contract_code").isin(selected_codes)
        )
        month_minute_df = read_silver("fact_futures_minute", minute_filter)
        if not month_minute_df.empty:
            minute_frames.append(month_minute_df)

    if minute_frames:
        minute_df = pd.concat(minute_frames, ignore_index=True)
        main_minute_df = minute_df.merge(
            mapping_df,
            how="inner",
            left_on=["exchange_code", "underlying_code", "trading_date", "contract_code"],
            right_on=["exchange_code", "underlying_code", "trading_date", "main_contract_code"],
            validate="many_to_one",
        )
        if main_minute_df.duplicated(["contract_code", "bar_at"]).any():
            raise ValueError("返回的分钟行情存在重复主键。")
        main_minute_df = main_minute_df.sort_values(
            ["bar_at", "contract_code"], kind="mergesort"
        ).reset_index(drop=True)
    else:
        main_minute_df = pd.DataFrame(columns=[
            *FUTURES_MINUTE_SCHEMA.names,
            *[column for column in mapping_df.columns
              if column not in FUTURES_MINUTE_SCHEMA.names],
        ])
    return mapping_df.reset_index(drop=True), main_minute_df

## 4. 执行并查看结果

In [5]:
main_contract_daily_df, main_minute_df = load_main_minute_bars(
    EXCHANGE_CODE,
    UNDERLYING_CODE,
    START_DATE,
    END_DATE,
    ROLL_TRIGGER_RATIO,
)

minute_counts_df = (
    main_minute_df.groupby("trading_date").size().rename("minute_count").reset_index()
    if not main_minute_df.empty
    else pd.DataFrame(columns=["trading_date", "minute_count"])
)
main_contract_coverage_df = main_contract_daily_df.merge(
    minute_counts_df, on="trading_date", how="left", validate="one_to_one"
)
main_contract_coverage_df["minute_count"] = (
    main_contract_coverage_df["minute_count"].fillna(0).astype(int)
)
print(
    f"{EXCHANGE_CODE}.{UNDERLYING_CODE} {START_DATE} ~ {END_DATE}: "
    f"主力交易日 {len(main_contract_daily_df)}，分钟记录 {len(main_minute_df)}"
)

XSGE.RB 2015-01-01 ~ 2026-09-01: 主力交易日 2835，分钟记录 998715


In [6]:
import numpy as np

# 1. 读取换月涉及的合约日收盘价
roll_daily_df = main_contract_daily_df.loc[
    main_contract_daily_df["is_roll"]
    & (main_contract_daily_df["trading_date"] > main_minute_df["trading_date"].iloc[0])
    & (main_contract_daily_df["trading_date"] <= main_minute_df["trading_date"].iloc[-1])
]

roll_contract_codes = roll_daily_df[
    ["previous_main_contract_code", "main_contract_code"]
].stack().unique()

daily_price_df = read_silver(
    FUTURES_DAILY_SCHEMA.metadata[b"table_name"].decode(),
    (ds.field("exchange_code") == EXCHANGE_CODE)
    & (ds.field("underlying_code") == UNDERLYING_CODE)
    & ds.field("contract_code").isin(roll_contract_codes)
    & (ds.field("trading_date") <= roll_daily_df["signal_trading_date"].max()),
)

daily_close_df = daily_price_df.loc[
    daily_price_df["has_market_data"] & (daily_price_df["close"] > 0)
].pivot(index="trading_date", columns="contract_code", values="close").sort_index()

# 2. 每次换月计算同日价差，再按日累加
roll_log_gap = pd.Series(0.0, index=main_contract_daily_df["trading_date"])

for roll in roll_daily_df.itertuples(index=False):
    common_close = daily_close_df.loc[
        :roll.signal_trading_date,
        [roll.previous_main_contract_code, roll.main_contract_code],
    ].dropna().iloc[-1]

    roll_log_gap.loc[roll.trading_date] = (
        np.log(common_close.iloc[1]) - np.log(common_close.iloc[0])
    )

cumulative_log_gap = main_minute_df["trading_date"].map(
    roll_log_gap.sort_index().cumsum()
)

# 3. 分别锚定首、尾 close
log_close = np.log(main_minute_df["close"])
main_minute_df["backward_adjusted_log_close"] = log_close - cumulative_log_gap
main_minute_df["forward_adjusted_log_close"] = (
    log_close - cumulative_log_gap + cumulative_log_gap.iloc[-1]
)

main_minute_df

In [ ]:
parquet_path = pathlib.Path(
    r"E:\Latitude_Analytics_v2\00_draft_collection_02\research_outputs\testing_10\rb_main_minute.parquet"
)
parquet_path.parent.mkdir(parents=True, exist_ok=True)
main_minute_df.to_parquet(parquet_path, index=False)